# AIMarx TRAIN-04 — public smoke A/B output evaluation

Run after the pilot notebook in the same free T4 runtime. The 20 smoke cases are repository-exposed, not blind. They are frozen for regression and cannot be used for later tuning reported on the same set. The model never receives gold or rubric.


In [ ]:
import os, pathlib, subprocess, sys
assert os.path.exists('/content'), 'Run in Google Colab'
subprocess.run(['nvidia-smi'], check=True)
repo = pathlib.Path('/content/AIMarx')
assert repo.exists(), 'Run the pilot notebook first in this runtime'
checkpoint = pathlib.Path('/content/aimarx-pilot-output/checkpoint-20')
assert checkpoint.exists(), 'Missing checkpoint-20; run the pilot notebook first'


In [ ]:
PINNED_COMMIT = '5b0f4330ac5aa3ecd9394de50c504d0fbe464147'
subprocess.run(['git', 'fetch', 'origin', 'codex/train03-domain-eval'], cwd=repo, check=True)
subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=repo, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip() == PINNED_COMMIT
os.chdir(repo)


## Prepare locked model inputs

`gold-after-review.jsonl` is written separately and is never passed to either model.


In [ ]:
subprocess.run([sys.executable, '-m', 'evals.training.train03.prepare', '/content/aimarx-train04-eval'], check=True)


## Generate base and adapter candidates with identical greedy decoding


In [ ]:
subprocess.run([sys.executable, '-m', 'evals.training.train03.generate', '--inputs', '/content/aimarx-train04-eval/inputs.jsonl', '--checkpoint', str(checkpoint), '--mode', 'base', '--output', '/content/aimarx-train04-base.jsonl'], check=True)
subprocess.run([sys.executable, '-m', 'evals.training.train03.generate', '--inputs', '/content/aimarx-train04-eval/inputs.jsonl', '--checkpoint', str(checkpoint), '--mode', 'adapter', '--output', '/content/aimarx-train04-adapter.jsonl'], check=True)


## Build the anonymous A/B review bundle

Review `candidate_A` and `candidate_B` before opening the separately downloaded after-review ZIP.


In [ ]:
subprocess.run([sys.executable, '-m', 'evals.training.train03.blind', '--inputs', '/content/aimarx-train04-eval/inputs.jsonl', '--base', '/content/aimarx-train04-base.jsonl', '--adapter', '/content/aimarx-train04-adapter.jsonl', '--output', '/content/aimarx-train04-review'], check=True)


In [ ]:
import shutil
review_zip = shutil.make_archive('/content/AIMarx-TRAIN04-review-first', 'zip', '/content/aimarx-train04-review', 'review.jsonl')
after_dir = pathlib.Path('/content/aimarx-train04-after-review')
after_dir.mkdir(exist_ok=False)
shutil.copy2('/content/aimarx-train04-review/reveal-after-review.json', after_dir)
shutil.copy2('/content/aimarx-train04-eval/gold-after-review.jsonl', after_dir)
after_zip = shutil.make_archive('/content/AIMarx-TRAIN04-open-after-review', 'zip', after_dir)
from google.colab import files
files.download(review_zip)
files.download(after_zip)
